# Chương 4 (FKG-E) đầy đủ trên Colab — BRSET 16.266 ảnh / 8.524 bệnh nhân

Notebook chạy gói `FKGE_BRSET_Tan_ext` (bản 02/11/2026) cho **toàn bộ thực nghiệm Chương 4**:

1. **Lượt chính** (`run.py`): FISA và FKG-E trên 5 outer fold; lưới 24 cấu hình
   (α {0,5; 0,7} × d_e {6, 8, 16} × λ_I {0,1; 0,5} × λ_P {0,3; 0,7}), tối đa 50 vòng, patience 10;
   chọn cấu hình trên validation theo bệnh nhân, đánh giá outer test một lần.
2. **Bổ sung** (`extra.py`): KB2 (S-FKGS), KB3–KB4 (đọc từ lưới), KB5 (K), KB6 (|R|), Ablation, Baseline
   (Node2Vec/TransE/DistMult + kNN, LR/RF/XGBoost).

Trước khi huấn luyện, cơ sở luật được làm sạch: luật mâu thuẫn (cùng tiền đề, khác nhãn) bị loại, mỗi luật trùng giữ
một bản. Chi tiết trong `README.md` và `VERIFICATION.md` của gói.

**Chuẩn bị:** tải `FKGE_BRSET_Tan_ext.zip` lên `MyDrive/`; chọn *Runtime → Change runtime type → GPU*; chạy lần lượt
các ô. Checkpoint, kết quả và log nằm trên Drive. Phiên bị ngắt (giới hạn thời gian của Colab) thì chạy lại từ đầu:
cấu hình/job đã xong được bỏ qua, cache teacher trong ổ tạm được tính lại (vài phút mỗi fold).

Khối lượng: lượt chính 24 cấu hình × 5 fold; bổ sung mỗi fold khoảng 14 lần huấn luyện FKG-E và 5 teacher. Số đo duy
nhất hiện có là trên CPU: d_e = 16 ở fold 1 mất khoảng 8 giây mỗi vòng. Thời gian trên GPU chưa được đo.

## 1. Cấu hình

| `MODE` | Việc làm |
|---|---|
| `"all"` | Lượt chính rồi bổ sung trên `OUTPUT` — **chạy đủ Chương 4** (mặc định). |
| `"extra_only"` | Chỉ bổ sung, trên output lượt chính đã có của **chính bản này** (output của bản khác bị từ chối vì cơ sở luật khác). |
| `"smoke"` | Kiểm tra phần mềm trên dữ liệu tổng hợp (vài phút). **Không phải kết quả BRSET.** |

`ONLY` chọn kịch bản bổ sung; `"kb3,kb4,baseline"` không cần huấn luyện FKG-E mới. Giữ nguyên `DEVICE` và `BLOCK_SIZE`
giữa các lần chạy tiếp.

In [ ]:
from pathlib import Path
MODE = "all"                     # "all" | "extra_only" | "smoke"
ZIP_PATH = Path("/content/drive/MyDrive/FKGE_BRSET_Tan_ext.zip")
OUTPUT = Path("/content/drive/MyDrive/FKGE_BRSET_Tan_ext_v3_16266")   # thư mục mới cho bản này; extra_only: output đã có
ONLY = "kb2,kb3,kb4,kb5,kb6,ablation,baseline"
FOLDS = [1, 2, 3, 4, 5]
DEVICE = "auto"                  # "auto" | "cuda" | "cpu"
BLOCK_SIZE = 128                 # giảm còn 64 nếu GPU thiếu bộ nhớ; giữ nguyên giá trị khi chạy tiếp
GPU_CHECK = True                 # chạy thử nhanh trên GPU (dữ liệu tổng hợp) trước lượt dài
WORKDIR = Path("/content")

## 2. Gắn Drive, giải nén, cài thư viện, kiểm tra gói

In [ ]:
import hashlib, json, os, subprocess, sys, time, zipfile
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Không chạy trên Colab: bỏ qua bước gắn Drive")

if MODE == "smoke":
    OUTPUT = OUTPUT.with_name(OUTPUT.name + "_SMOKE")
    FOLDS = [1, 2, 3]            # dữ liệu tổng hợp quá nhỏ cho fold 4–5
assert ZIP_PATH.exists(), f"Chưa thấy {ZIP_PATH}: tải FKGE_BRSET_Tan_ext.zip lên Drive"
with zipfile.ZipFile(ZIP_PATH) as z:
    assert all((WORKDIR / n).resolve().is_relative_to(WORKDIR.resolve()) for n in z.namelist()), "Đường dẫn lạ trong zip"
    z.extractall(WORKDIR)
bundle = WORKDIR / "FKGE_BRSET_Tan_ext"
print("ZIP sha256:", hashlib.sha256(ZIP_PATH.read_bytes()).hexdigest())
release = json.loads((bundle / "RELEASE.json").read_text())
print({k: release[k] for k in ("code_commit", "images", "patients", "outer_folds")}, release.get("extension"))
LOGS = OUTPUT.parent / (OUTPUT.name + "_logs")
LOGS.mkdir(parents=True, exist_ok=True)

def run_logged(args, log_name):
    """Chạy lệnh, in từng dòng ra notebook và ghi thêm vào log trên Drive; lỗi thì dừng ô."""
    command = [sys.executable, "-B", "-X", "utf8", "-u", *map(str, args)]
    print("Lệnh:", " ".join(command), flush=True)
    with open(LOGS / log_name, "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} {' '.join(command)}\n")
        process = subprocess.Popen(command, cwd=bundle, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, encoding="utf-8", errors="replace", bufsize=1)
        for line in process.stdout:
            log.write(line)
            if not line.startswith("EPOCH"):      # log đầy đủ trên Drive; notebook bớt dòng epoch
                print(line, end="", flush=True)
        code = process.wait()
    if code:
        raise RuntimeError(f"Lệnh lỗi (mã {code}); xem {LOGS / log_name}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(bundle / "requirements-colab.txt")], check=True)
run_logged(["run.py", "--check"], "check.log")
run_logged(["-m", "unittest", "discover", "-s", "tests"], "tests.log")

## 3. Phần cứng và chạy thử nhanh trên GPU

Chạy thử toàn bộ luồng (teacher, FKG-E, KB2, KB5, baseline) trên dữ liệu tổng hợp bằng đúng thiết bị sẽ dùng, để lỗi
CUDA/bộ nhớ lộ ra sau vài phút thay vì giữa lượt dài. Kết quả chạy thử ghi vào ổ tạm, không vào `OUTPUT`.

In [ ]:
import shutil
import torch, psutil
print("PyTorch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
      f"| RAM {psutil.virtual_memory().total / 2**30:.1f} GiB | đĩa tạm trống {psutil.disk_usage('/tmp').free / 2**30:.0f} GiB")
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("Chưa chọn GPU runtime")
device = "cuda" if DEVICE == "auto" and torch.cuda.is_available() else ("cpu" if DEVICE == "auto" else DEVICE)
if device == "cpu" and MODE != "smoke":
    print("CẢNH BÁO: đang chạy trên CPU — chậm hơn nhiều so với GPU.")
if GPU_CHECK and MODE != "smoke" and device == "cuda":
    check = Path("/tmp/fkge_gpu_check")
    shutil.rmtree(check, ignore_errors=True)
    run_logged(["run.py", "--smoke", "--folds", 1, "--device", "cuda", "--block-size", BLOCK_SIZE, "--output", check],
               "gpu_check.log")
    run_logged(["extra.py", "--output", check, "--folds", 1, "--only", "kb2,kb5,baseline", "--device", "cuda",
                "--block-size", BLOCK_SIZE], "gpu_check.log")
    print("Chạy thử GPU: đạt")

## 4. Tiến độ

Chạy ô này bất cứ lúc nào (kể cả sau khi phiên bị ngắt) để xem mỗi fold đã xong bao nhiêu cấu hình và job.

In [ ]:
def progress():
    sys.path.insert(0, str(bundle))
    from extra import SCENARIOS, job_key, plan
    grid = json.loads((bundle / "config.json").read_text())["grid"]
    n_trials = 1
    for values in grid.values():
        n_trials *= len(values)
    if MODE == "smoke":
        n_trials = 2
    ecfg = json.loads((bundle / "config_extra.json").read_text())
    scenarios = [s.strip() for s in ONLY.split(",") if s.strip()]
    for f in FOLDS:
        d = OUTPUT / f"fold_{f:02}"
        trials = len(list((d / "trials").glob("trial_[0-9][0-9][0-9].json")))
        line = f"fold {f}: chuẩn bị {'xong' if (d / 'preparation.json').exists() else 'chưa'} | lưới {trials}/{n_trials}"
        line += f" | lượt chính {'xong' if (d / 'results.json').exists() else 'chưa'}"
        if (d / "selected.json").exists():
            todo = plan(d, scenarios, ecfg)
            done = sum((OUTPUT / "extra" / "jobs" / d.name / f"{job_key(j)}.json").exists() for j in todo)
            line += f" | bổ sung {done}/{len(todo)} job"
        print(line)

progress()

## 5. Lượt chính (`run.py`)

Bỏ qua khi `MODE = "extra_only"`. Nếu `OUTPUT` đã được tạo bằng **mã khác** (gói gốc hay bản trước), ô này dừng
với lỗi: hãy đặt `OUTPUT` sang thư mục mới (cơ sở luật và lưới đã đổi, không trộn kết quả).

In [ ]:
def code_hashes():
    return {p.relative_to(bundle).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest()
            for p in sorted(bundle.rglob("*.py"))
            if not any(part in {"outputs", "private", "__pycache__"} for part in p.relative_to(bundle).parts)}

manifest_path = OUTPUT / "run_manifest.json"
same_code = (not manifest_path.exists()
             or json.loads(manifest_path.read_text())["identity"]["code_sha256"] == code_hashes())
if MODE == "extra_only":
    assert manifest_path.exists(), f"{OUTPUT} chưa có run_manifest.json của lượt chính"
    print("extra_only: không chạy lượt chính")
elif not same_code:
    raise RuntimeError(f"{OUTPUT} được tạo bằng mã khác: đặt OUTPUT sang thư mục mới cho bản này")
else:
    main_args = ["run.py", "--device", DEVICE, "--block-size", BLOCK_SIZE, "--output", OUTPUT, "--resume",
                 "--folds", *FOLDS] + (["--smoke"] if MODE == "smoke" else [])
    run_logged(main_args, "run_main.log")

done = [f for f in FOLDS if (OUTPUT / f"fold_{f:02}" / "results.json").exists()]
for f in FOLDS:
    trials = len(list((OUTPUT / f"fold_{f:02}" / "trials").glob("trial_*.json")))
    print(f"fold {f}: {'xong' if f in done else 'chưa xong'} ({trials} cấu hình đã huấn luyện)")

## 6. Thí nghiệm bổ sung (`extra.py`)

Chỉ chạy trên các fold đã xong lượt chính. Mỗi job lưu ngay vào `OUTPUT/extra/jobs/`; chạy lại ô này để tiếp tục.
Bảng ghi `INCOMPLETE` khi chưa đủ 5 fold — khi đó không dùng số liệu để báo cáo.

In [ ]:
assert done, "Chưa fold nào xong lượt chính"
extra_args = ["extra.py", "--output", OUTPUT, "--device", DEVICE, "--block-size", BLOCK_SIZE,
              "--only", ONLY, "--folds", *done]
if (OUTPUT / "extra" / "extra_manifest.json").exists():
    extra_args.append("--resume")
run_logged(extra_args, "run_extra.log")
progress()

## 7. Xem kết quả

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display
pd.set_option("display.width", 200, "display.max_columns", 30)
main_reports, extra_reports = OUTPUT / "reports", OUTPUT / "extra" / "reports"
display(Markdown("### Lượt chính"))
for name in ["fold_population.png", "metrics_by_fold.png", "metrics_mean_std.png", "roc_pr.png"]:
    if (main_reports / name).exists():
        display(Image(filename=str(main_reports / name)))
display(Markdown("### Bổ sung Chương 4"))
if (extra_reports / "README.txt").exists():
    print((extra_reports / "README.txt").read_text(encoding="utf-8"))
for name in ["baseline", "ablation", "kb2_fkgs", "kb3_d_e", "kb4_lambda", "profile_alpha", "kb5_K", "kb6_rules"]:
    if (extra_reports / f"{name}_mean_std.png").exists():
        display(Image(filename=str(extra_reports / f"{name}_mean_std.png")))
for name in ["kb4_heatmap.png", "kb6_time.png"]:
    if (extra_reports / name).exists():
        display(Image(filename=str(extra_reports / name)))
if (extra_reports / "baseline_pooled_oof.csv").exists():
    display(Markdown("Baseline — chỉ số gộp ngoài fold (khác trung bình theo fold):"))
    display(pd.read_csv(extra_reports / "baseline_pooled_oof.csv")[
        ["method", "n_folds", "roc_auc", "average_precision", "balanced_accuracy", "sensitivity", "specificity"]].round(4))

## 8. Đóng gói để gửi lại

In [ ]:
stamp = time.strftime("%Y%m%d_%H%M")
package = OUTPUT.parent / f"{OUTPUT.name}_bao_cao_{stamp}.zip"
with zipfile.ZipFile(package, "w", zipfile.ZIP_DEFLATED) as z:
    for path in [*main_reports.rglob("*"), *extra_reports.rglob("*"), *LOGS.rglob("*"),
                 OUTPUT / "run_manifest.json", OUTPUT / "environment-freeze.txt",
                 OUTPUT / "extra" / "extra_manifest.json",
                 *OUTPUT.glob("fold_*/selected.json"), *OUTPUT.glob("fold_*/results.json"),
                 *OUTPUT.glob("fold_*/preparation.json")]:
        if path.is_file():
            z.write(path, path.relative_to(OUTPUT.parent))
print("Gửi file:", package, f"({package.stat().st_size / 2**20:.1f} MiB)")